In [7]:
import numpy as np
import pandas as pd
from joblib import load
from utils.paths import DATASETS_PREPROCESSED_DIR, PROJECT_ROOT
import dieboldmariano

pp_models_dir = PROJECT_ROOT / "power_prediction" / "models"
ar_models_dir = PROJECT_ROOT / "autoregression-models" / "models"

alpha = 0.05

In [8]:
TARGET_LAGS = [1, 2, 3, 4, 5, 6, 24, 48]
FEATURE_LAGS = {
    "temperature_2m": [1, 17],
    "relative_humidity_2m": [6, 30],
    "wind_speed_10m": [6, 30],
    "wind_gusts_10m": [6, 30],
}


def add_lags(df, horizon):
    lagged = df.copy()
    lagged["hour"] = pd.to_datetime(lagged["time"]).dt.hour
    lagged["dayofweek"] = pd.to_datetime(lagged["time"]).dt.dayofweek
    lagged["month"] = pd.to_datetime(lagged["time"]).dt.month
    lagged["row_idx"] = np.arange(len(lagged))

    for lag in TARGET_LAGS:
        lagged[f"delta_p_lag_{lag}"] = lagged["delta_p"].shift(lag)

    for feature, lags in FEATURE_LAGS.items():
        for lag in lags:
            lagged[f"{feature}_lag_{lag}"] = lagged[feature].shift(lag)

    lagged["target"] = lagged["delta_p"].shift(-horizon)
    lagged = lagged.dropna().reset_index(drop=True)

    return lagged

In [9]:
rows = []

for loc in range(1, 5):
    df = pd.read_csv(DATASETS_PREPROCESSED_DIR / f"Location_{loc}_test.csv")

    X_pp = df.drop(columns=["time", "p_base", "delta_p"])
    y_pp_all = load(pp_models_dir / f"stacking_model{loc}.pkl").predict(X_pp)

    for horizon in [24, 48]:
        lagged = add_lags(df, horizon)
        ar_model = load(ar_models_dir / f"stacking_regressor_h{horizon}.pkl")

        X_ar = lagged.drop(columns=["time", "delta_p", "target", "p_base", "row_idx"])
        y = lagged["target"].to_numpy()
        y_ar = ar_model.predict(X_ar)
        y_pp = np.asarray(y_pp_all)[lagged["row_idx"].to_numpy() + horizon]  # getting only the predictions corresponding to the lagged rows

        dm_stat, p_value = dieboldmariano.dm_test(y, y_ar, y_pp, h=horizon)

        rows.append({
            "location": f"Location_{loc}", 
            "horizon": horizon, 
            "dm_stat": dm_stat, 
            "p_value": p_value, 
            "reject_h0": p_value < alpha
            })

pd.DataFrame(rows)

/home/kuba/Studia/IIst/ProjektNaukowoWdrozeniowy/projekt/ProjektNaukowoWdrozeniowy/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/kuba/Studia/IIst/ProjektNaukowoWdrozeniowy/projekt/ProjektNaukowoWdrozeniowy/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/kuba/Studia/IIst/ProjektNaukowoWdrozeniowy/projekt/ProjektNaukowoWdrozeniowy/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/kuba/Studia/IIst/ProjektNaukowoWdrozeniowy/projekt/ProjektNaukowoWdrozeniowy/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature name

,location,horizon,dm_stat,p_value,reject_h0
0,Location_1,24,0.034440,0.972527,False
1,Location_1,48,0.956956,0.338616,False
2,Location_2,24,3.348430,0.000816,True
3,Location_2,48,4.648535,0.000003,True
4,Location_3,24,-4.248412,0.000022,True
5,Location_3,48,-0.432387,0.665471,False
6,Location_4,24,3.008585,0.002632,True
7,Location_4,48,2.898784,0.003756,True
